In [ ]:
import torch
import torchxrayvision as xrv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
import random
import gc
from IPython.display import display, clear_output

# Local imports
from monai.networks.nets import DiffusionModelUNet
from monai.networks.schedulers import DDPMScheduler
from datasets.nih_dataset import get_nih_loaders

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

CSV_PATH = "./NIH_Chest_XRay/Data_Entry_2017.csv"
IMG_DIR = "./NIH_Chest_XRay/images"
DIFFUSION_WEIGHTS = "weights/old/denoiser_res_512_epoch_20.pt" # Update to your newly trained weights

NUM_IMAGES = 20
PURIFY_TIMESTEP = 50

print(f"Hardware: {torch.cuda.get_device_name(0)} (8GB VRAM Optimized)")
print(f"Precision: {DTYPE} | Target Sample: {NUM_IMAGES} images")

In [ ]:
def get_diffusion_stack(res=512):
    channels = (64, 128, 256, 512)
    att_levels = (False, False, False, True)
    model = DiffusionModelUNet(
        spatial_dims=2, in_channels=1, out_channels=1,
        channels=channels, attention_levels=att_levels,
        num_res_blocks=2, num_head_channels=64,
    )
    scheduler = DDPMScheduler(num_train_timesteps=1000, beta_start=0.0001, beta_end=0.02, schedule="linear_beta")
    return model, scheduler

# 1. Load Denoiser
denoiser, scheduler = get_diffusion_stack(res=512)
denoiser.load_state_dict(torch.load(DIFFUSION_WEIGHTS, map_location=DEVICE))
denoiser.to(DEVICE).eval()
# Now that we use [0,1] logic, clipping is mathematically safe and recommended
scheduler.clip_sample = True 

# 2. Load Classifier
classifier = xrv.models.DenseNet(weights="densenet121-res224-nih")
classifier.to(DEVICE).eval()
xrv_pathologies = classifier.pathologies

print("✅ Models loaded into VRAM.")

In [ ]:
# Load at 1024px to preserve max original detail
loaders, nih_pathologies = get_nih_loaders(CSV_PATH, IMG_DIR, batch_size=1, resize_to=1024)
test_loader = loaders['test']

# Match indices to ensure perfect alignment
indices = [xrv_pathologies.index(p) for p in nih_pathologies if p in xrv_pathologies]
matched_pathologies = [xrv_pathologies[i] for i in indices]

# Extract random subset
dataset_length = len(test_loader.dataset)
random_indices = random.sample(range(dataset_length), NUM_IMAGES)

print(f"✅ Selected {NUM_IMAGES} random indices for visualization.")

In [ ]:
def plot_results(img_orig, img_denoised, preds_orig, preds_denoised, labels, idx):
    """Generates a 1x3 grid: Original, Denoised, and a Probability Bar Chart."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Original Image
    axes[0].imshow(img_orig[0, 0].cpu().numpy(), cmap='gray', vmin=-1024, vmax=1024)
    axes[0].set_title(f"Original (1024px -> 224px)\nScale: [-1024, 1024]")
    axes[0].axis('off')
    
    # 2. Denoised Image
    axes[1].imshow(img_denoised[0, 0].cpu().numpy(), cmap='gray', vmin=-1024, vmax=1024)
    axes[1].set_title(f"Denoised (t={PURIFY_TIMESTEP})\nScale: [-1024, 1024]")
    axes[1].axis('off')
    
    # 3. Probability Chart
    y_pos = np.arange(len(matched_pathologies))
    width = 0.35
    
    axes[2].barh(y_pos - width/2, preds_orig, width, label='Original', color='skyblue')
    axes[2].barh(y_pos + width/2, preds_denoised, width, label='Denoised', color='coral')
    
    # Highlight true labels with a star
    labels_idx = np.where(labels[0] == 1)[0]
    for l_idx in labels_idx:
        axes[2].text(1.05, l_idx, '★ TRUE', color='red', va='center', fontweight='bold')
        
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels(matched_pathologies)
    axes[2].set_xlim(0, 1.1)
    axes[2].axvline(0.5, color='gray', linestyle='--')
    axes[2].set_title(f"DenseNet Predictions (Sample {idx+1}/{NUM_IMAGES})")
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()

# --- Execution ---
metrics_log = []

with torch.inference_mode():
    for current_idx, ds_idx in enumerate(random_indices):
        # Fetch specific item
        img_1024, label = test_loader.dataset[ds_idx]
        img_1024 = img_1024.unsqueeze(0).to(DEVICE) # Add batch dim
        label = label.unsqueeze(0).numpy()
        
        # --- PATH A: Original Inference ---
        img_orig_224 = F.interpolate(img_1024, size=(224, 224), mode='bilinear')
        logits_orig = classifier(img_orig_224)
        preds_orig = torch.sigmoid(logits_orig)[:, indices].cpu().numpy()[0]
        
        # --- PATH B: Denoising Pipeline ---
        img_512 = F.interpolate(img_1024, size=(512, 512), mode='bilinear')
        
        # THE FIX: Scale to [0, 1] for Diffusion Math
        img_512_norm = (img_512 + 1024.0) / 2048.0
        
        with torch.amp.autocast(device_type='cuda', dtype=DTYPE):
            t_start = torch.full((1,), PURIFY_TIMESTEP, device=DEVICE).long()
            noise = torch.randn_like(img_512_norm)
            curr_img = scheduler.add_noise(img_512_norm, noise, t_start)
            
            scheduler.set_timesteps(num_inference_steps=50)
            p_steps = [t for t in scheduler.timesteps if t <= PURIFY_TIMESTEP]
            
            for t in p_steps:
                t_batch = torch.full((1,), t, device=DEVICE).long()
                model_out = denoiser(curr_img, t_batch)
                curr_img = scheduler.step(model_out, t, curr_img)[0]
                
        # THE FIX: Scale back to [-1024, 1024] for Classifier
        img_denoised_clinical = (curr_img * 2048.0) - 1024.0
        img_denoised_224 = F.interpolate(img_denoised_clinical, size=(224, 224), mode='bilinear')
        
        logits_denoised = classifier(img_denoised_224)
        preds_denoised = torch.sigmoid(logits_denoised)[:, indices].cpu().numpy()[0]
        
        # --- Log & Plot ---
        abs_diff = np.mean(np.abs(preds_orig - preds_denoised))
        metrics_log.append(abs_diff)
        
        plot_results(img_orig_224, img_denoised_224, preds_orig, preds_denoised, label, current_idx)
        
        # VRAM Cleanup
        del img_1024, img_orig_224, img_512, img_512_norm, curr_img, img_denoised_clinical, img_denoised_224
        torch.cuda.empty_cache()

In [ ]:
print("--- Experiment Summary ---")
print(f"Total Images Analyzed: {NUM_IMAGES}")
print(f"Average Absolute Probability Shift: {np.mean(metrics_log):.4f}")

if np.mean(metrics_log) > 0.05:
    print("Result: The denoiser is causing significant shifts (>5%) in the classifier's confidence.")
else:
    print("Result: The denoiser is maintaining high structural fidelity with minor confidence shifts.")